# 动手练习：小世界与无标度网络模型

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。

## 任务 1：环形规则网格的聚集系数与路径长度

环形规则网格中每个节点与左右各 $K/2$ 个最近邻相连。专题页推导出
$C=\dfrac{3(K-2)}{4(K-1)}$，平均路径长度 $L\approx N/(2K)$。

对 $N=200$、$K=4,6,8,10$：

1. 用公式计算 `C_formula`（字典，键为 K）；
2. 用 `nx.watts_strogatz_graph(N, K, 0)`（$\beta=0$ 即规则网格）求实际的平均聚集系数 `C_sim` 与平均路径长度 `L_sim`（都是字典）。

期望看到：`C_formula` 与 `C_sim` 完全一致，$K=10$ 时为 $2/3$；`L_sim` 略大于 $N/(2K)$。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

N_ring = 200
K_list = [4, 6, 8, 10]

In [ ]:
C_formula, C_sim, L_sim = {}, {}, {}
for K in K_list:
    ring = nx.watts_strogatz_graph(N_ring, K, 0)
    pass   # TODO: 填写 C_formula[K]、C_sim[K]、L_sim[K]

In [ ]:
# 核对（不用修改）
assert set(C_formula) == set(K_list) and set(C_sim) == set(K_list) and set(L_sim) == set(K_list), "三个字典都应包含 K = 4, 6, 8, 10"
assert abs(C_formula[4] - 0.5) < 1e-9 and abs(C_formula[10] - 2 / 3) < 1e-9, "公式 C = 3(K-2)/(4(K-1))：K=4 时 0.5，K=10 时 2/3"
for K in K_list:
    assert abs(C_sim[K] - C_formula[K]) < 1e-6, "K=%d 时实际聚集系数应与公式一致；检查重连概率是否为 0" % K
    assert 1.0 <= L_sim[K] / (N_ring / (2 * K)) < 1.1, "K=%d 时 L 应略大于 N/(2K)；检查是否用了 average_shortest_path_length" % K
print("✓ 任务 1 通过")

## 任务 2：复现 Watts–Strogatz 的经典曲线

对一组重连概率 $\beta$（0.001 到 1，对数均匀取 10 个），生成 $N=300$、$K=10$ 的 WS 网络（`connected_watts_strogatz_graph`，种子 42），计算 $C(\beta)/C(0)$ 和 $L(\beta)/L(0)$，存为列表 `C_ratio`、`L_ratio`，画在对数横轴上。

期望看到：$\beta\approx0.01$ 时 $L$ 已降到初始值的一半左右，而 $C$ 仍接近初始值；$\beta=1$ 时两者都很低。两条曲线之间的区间就是小世界区间。

> **提示：** “破坏聚类需要大量随机性，制造捷径只需要一点随机性。”

In [ ]:
# 准备（直接运行）
N_ws, K_ws = 300, 10
betas = np.logspace(-3, 0, 10)
G0 = nx.watts_strogatz_graph(N_ws, K_ws, 0)
C0 = nx.average_clustering(G0)
L0 = nx.average_shortest_path_length(G0)

In [ ]:
C_ratio, L_ratio = [], []
for b in betas:
    Gb = nx.connected_watts_strogatz_graph(N_ws, K_ws, b, seed=42)
    pass   # TODO: 追加 C(β)/C0 和 L(β)/L0

In [ ]:
# 核对（不用修改）
C_ratio, L_ratio = np.asarray(C_ratio, dtype=float), np.asarray(L_ratio, dtype=float)
assert C_ratio.shape == (10,) and L_ratio.shape == (10,), "C_ratio、L_ratio 各应有 10 个值"
i01 = int(np.argmin(np.abs(betas - 0.01)))
assert C_ratio[i01] > 0.9, "β≈0.01 时 C 应仍接近初始值（>0.9）；检查是否除以了 C0"
assert L_ratio[i01] < 0.6, "β≈0.01 时 L 应已大幅下降；检查是否除以了 L0"
assert C_ratio[-1] < 0.15 and L_ratio[-1] < 0.3, "β=1 时网络接近随机，C 和 L 都应很低"
assert np.all(C_ratio <= 1.02) and np.all(L_ratio <= 1.02), "比值不应明显大于 1"
print("✓ 任务 2 通过")

## 任务 3：BA 网络与 ER 网络的度分布（CCDF）

生成 $N=3000$、$m=2$ 的 BA 网络（种子 1），以及平均度相同的 ER 网络（种子 1）。写函数 `ccdf(degrees)`，返回 `(ks, p)`：`ks` 是出现过的度值（升序），`p[i]` 是度 $\ge$ `ks[i]` 的节点比例 $P(K\ge k)$。在双对数坐标下画两者的 CCDF，并记录最大度 `kmax_ba`、`kmax_er`。

期望看到：BA 的 CCDF 尾部近似直线（幂律的 CCDF 斜率为 $-(\gamma-1)\approx-2$），最大度超过 100；ER 的 CCDF 很快“掉崖”，最大度只有十几。

> **提示：** `np.array([(degrees >= k).mean() for k in ks])`。

In [ ]:
# 准备（直接运行）
G_ba = nx.barabasi_albert_graph(3000, 2, seed=1)
avg_k = 2 * G_ba.number_of_edges() / 3000
G_er = nx.gnp_random_graph(3000, avg_k / 2999, seed=1)
deg_ba = np.array([d for _, d in G_ba.degree()])
deg_er = np.array([d for _, d in G_er.degree()])

In [ ]:
def ccdf(degrees):
    degrees = np.asarray(degrees)
    ks = np.unique(degrees)
    p = None   # TODO
    return ks, p


kmax_ba = None   # TODO
kmax_er = None   # TODO

In [ ]:
# 核对（不用修改）
ks_t, p_t = ccdf([1, 1, 2, 3, 3, 3])
assert p_t is not None, "ccdf 还没有返回 p"
assert np.allclose(ks_t, [1, 2, 3]) and np.allclose(p_t, [1, 4 / 6, 3 / 6]), "对度序列 [1,1,2,3,3,3]，P(K≥1)=1, P(K≥2)=4/6, P(K≥3)=3/6；检查是否用了 >= 而不是 =="
ks_b, p_b = ccdf(deg_ba)
assert abs(p_b[0] - 1) < 1e-12 and np.all(np.diff(p_b) < 0), "CCDF 从 1 开始，并随 k 严格下降"
assert kmax_ba == deg_ba.max() and kmax_er == deg_er.max(), "kmax_ba、kmax_er 应为两个网络的最大度"
assert kmax_ba > 5 * kmax_er, "BA 网络应有远大于 ER 的枢纽"
print("✓ 任务 3 通过")

## 任务 4：用最大似然估计度指数

对任务 3 的 BA 度序列，用离散修正的最大似然估计
$$\hat\gamma = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}-1/2}\right]^{-1}$$
（只用 $k_i\ge k_{\min}$ 的 $n$ 个节点）。依次取 $k_{\min}=2,4,6,8,10$，把估计值存入字典 `gamma_hat`；再用“线性分箱直方图 + 双对数最小二乘”得到 `gamma_ls`，与理论值 3 比较。

期望看到：$k_{\min}$ 越大，MLE 越接近 3（BA 的度分布只在大 $k$ 处才趋于 $k^{-3}$）；最小二乘结果明显偏离 3，因为尾部大量只出现一次的度值把拟合拉偏了。

> **提示：** 最小二乘用 `vals, counts = np.unique(k, return_counts=True)` 和 `np.polyfit(np.log(vals), np.log(counts / counts.sum()), 1)`，斜率取负号。

In [ ]:
# 准备（直接运行）
k_ba = deg_ba.astype(float)

In [ ]:
def gamma_mle(k, k_min):
    kk = k[k >= k_min]
    return None   # TODO


gamma_hat = {}    # TODO: {k_min: gamma_mle(k_ba, k_min)}
gamma_ls = None   # TODO

In [ ]:
# 核对（不用修改）
assert gamma_mle(np.array([2.0, 4.0, 8.0]), 2) is not None, "gamma_mle 还没有返回值"
assert abs(gamma_mle(np.array([2.0, 4.0, 8.0]), 2) - (1 + 3 / np.sum(np.log(np.array([2, 4, 8]) / 1.5)))) < 1e-9, "公式不对：分母里是 ln(k_i / (k_min - 0.5)) 的和"
assert set(gamma_hat) == {2, 4, 6, 8, 10}, "gamma_hat 应包含 k_min = 2, 4, 6, 8, 10"
assert gamma_hat[2] < gamma_hat[6] < gamma_hat[10], "随 k_min 增大，估计值应逐渐上升、接近 3"
assert 2.6 < gamma_hat[8] < 3.2, "k_min=8 时 MLE 应约为 2.8；检查是否只用了 k >= k_min 的节点"
assert gamma_ls is not None and abs(gamma_ls - 3) > 0.5, "直方图最小二乘应明显偏离 3（约 1.9）；检查斜率是否取了负号"
print("✓ 任务 4 通过")

## 任务 5：朋友悖论与 Molloy–Reed 判据

沿一条随机边到达的节点，其度为 $k$ 的概率正比于 $k\,p_k$，所以“随机邻居”的平均度是 $\langle k^2\rangle/\langle k\rangle$，总不小于 $\langle k\rangle$。

对任务 3 的 BA 和 ER 网络分别计算：

1. 平均度 `km` 与随机邻居的平均度 `knn`（字典，键为 `"BA"`、`"ER"`）；做法是把每条边的两个端点的度都收集起来再取平均，并与 $\langle k^2\rangle/\langle k\rangle$ 核对；
2. Molloy–Reed 比值 `mr` $=\langle k^2\rangle/\langle k\rangle$，大于 2 时存在巨连通分支。

期望看到：ER 的 `knn` 约为 `km + 1`（泊松分布 $\langle k^2\rangle=\langle k\rangle^2+\langle k\rangle$）；BA 的 `knn` 约为 `km` 的 3 倍。

In [ ]:
# 准备（直接运行）
nets = {"BA": G_ba, "ER": G_er}

In [ ]:
km, knn, mr = {}, {}, {}
for name, H in nets.items():
    pass   # TODO: 填写 km[name]、knn[name]（按边端点求平均）、mr[name]

In [ ]:
# 核对（不用修改）
for name, H in nets.items():
    assert name in km and name in knn and name in mr, "km、knn、mr 都要包含 %s" % name
    kv = np.array([d for _, d in H.degree()], dtype=float)
    assert abs(km[name] - kv.mean()) < 1e-9, "%s 的平均度不对" % name
    assert abs(knn[name] - (kv ** 2).sum() / kv.sum()) < 1e-6, "%s 的随机邻居平均度应等于 <k²>/<k>；检查是否把每条边的两个端点都算上了" % name
    assert abs(mr[name] - knn[name]) < 1e-6, "Molloy–Reed 比值就是 <k²>/<k>"
    assert mr[name] > 2, "两个网络的平均度都约为 4，都满足 <k²>/<k> > 2"
assert abs(knn["ER"] - km["ER"] - 1) < 0.15, "ER（泊松）网络中随机邻居的平均度约为 <k>+1"
assert knn["BA"] > 2 * km["BA"], "BA 网络中随机邻居的平均度应远大于平均度"
print("✓ 任务 5 通过")

## 任务 6：小分析：《悲惨世界》人物网络是小世界吗

计算小世界系数 $\sigma = \dfrac{C/C_{\text{rand}}}{L/L_{\text{rand}}}$，分别用两种零模型：

- (a) 同节点数、同边数的 ER 网络；
- (b) 保持度序列的随机化网络：对原网络做约 $10M$ 次双边交换（`nx.double_edge_swap`，`max_tries` 设大一些）。

每种零模型生成 5 个（种子 0–4），在最大连通分支上计算 $C$ 和 $L$ 后取平均，结果存为 `sigma_er` 和 `sigma_dp`。再用一两句话比较：哪种零模型更严格？为什么保度随机化后 $C_{\text{rand}}$ 更高？
本任务是开放的，只检查取值范围；也可以改用别的网络或更多随机样本。

In [ ]:
# 准备（直接运行）
G_les = nx.Graph()
G_les.add_edges_from(nx.les_miserables_graph().edges())
n_les, m_les = G_les.number_of_nodes(), G_les.number_of_edges()
C_les = nx.average_clustering(G_les)
L_les = nx.average_shortest_path_length(G_les)


def lcc(H):
    return H.subgraph(max(nx.connected_components(H), key=len)).copy()

In [ ]:
sigma_er = None   # TODO
sigma_dp = None   # TODO

In [ ]:
# 核对（不用修改）
assert sigma_er is not None and sigma_dp is not None, "sigma_er、sigma_dp 还没有赋值"
assert np.isfinite(sigma_er) and np.isfinite(sigma_dp), "σ 应为有限的数；检查是否在最大连通分支上计算了 L"
assert 0 < sigma_er < 50 and 0 < sigma_dp < 50, "σ 应为正数且量级合理；检查 C/C_rand 与 L/L_rand 是否写反"
print("✓ 任务 6 通过")